In [21]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import GridSearchCV
#Here I will be using the in built Sklearn Diabetes dataset to carry out a regression model and practice some hyperparameter tuning.

db = datasets.load_diabetes()

x = db.data
y = db.target

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=500 #No random shuffling of data, the shuffling will be in a specific order to recreate the same workflow.
)

#POLY = PolynomialFeatures()
#x = POLY.fit_transform(x)
#Including polynomial features harms my r2 score most likely due to overfitting or multicollinearity.

models = {

    "Linear Regression":
        LinearRegression(),

    #Random Forest#

    "RF Baseline":
        RandomForestRegressor(
            random_state=500,
            n_jobs=-1
        ),

    "RF More Trees":
        RandomForestRegressor(
            n_estimators=500,
            random_state=500,
            n_jobs=-1
        ),

    "RF Limited Depth":
        RandomForestRegressor(
            n_estimators=300,
            max_depth=5,
            random_state=500,
            n_jobs=-1
        ),

    "RF Small Leaves":
        RandomForestRegressor(
            n_estimators=300,
            min_samples_leaf=5,
            random_state=500,
            n_jobs=-1
        ),

    #Hist Gradient Bossting#

    "GB Baseline":
        HistGradientBoostingRegressor(
            random_state=500
        ),

    "GB Lower Learning Rate":
        HistGradientBoostingRegressor(
            learning_rate=0.05,
            max_iter=300,
            random_state=500
        ),

    "GB Shallow Trees":
        HistGradientBoostingRegressor(
            max_depth=3,
            learning_rate=0.1,
            random_state=500
        ),

    "GB Larger Leaves":
        HistGradientBoostingRegressor(
            min_samples_leaf=20,
            learning_rate=0.1,
            random_state=500
        )
}

for i, model in models.items():
    model.fit(x_train, y_train)
    Y_PRED = model.predict(x_test)
    R2 = r2_score(y_test, Y_PRED)
    print(i, R2)


#Ridge instead of plain Linear Regression
#Ridge = Linear Regression + a penalty that shrinks coefficients slightly, This can help when the plain model is slightly overfitting

ridge_model = Ridge()

#range of alpha values (the "strength" of the penalty)
#GridSearchCV tests each one using cross-validation and picks the best
param_grid = {'alpha': [0.01, 0.05, 0.1, 0.15, 0.2, 0.25]}

grid_search = GridSearchCV(ridge_model, param_grid, cv=5, scoring='r2')
grid_search.fit(x_train, y_train)

print("Best alpha:", grid_search.best_params_)

best_ridge = grid_search.best_estimator_
Y_PRED = best_ridge.predict(x_test)
R2 = r2_score(y_test, Y_PRED)

print("Ridge R2:", R2)
print("Baseline with PolynomialFeatures 0.4588432773714499")
print("From my testing I can see that Linear regression is the best model even after testing some different settings on the HGB and RF and this may be due to the fact that the dataset is quite small with only 442 observations, 10 features and the signals are p[pretty linear")

Linear Regression 0.5070867672483779
RF Baseline 0.3673025734417876
RF More Trees 0.396491732368786
RF Limited Depth 0.4178321202611738
RF Small Leaves 0.42775122228344586
GB Baseline 0.30201043722149223
GB Lower Learning Rate 0.2972097418445543
GB Shallow Trees 0.3237169225857762
GB Larger Leaves 0.30201043722149223
Best alpha: {'alpha': 0.1}
Ridge R2: 0.4928643145751168
Baseline with PolynomialFeatures 0.4588432773714499
From my testing I can see that Linear regression is the best model even after testing some different settings on the HGB and RF and this may be due to the fact that the dataset is quite small with only 442 observations, 10 features and the signals are p[pretty linear
